In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch

# Setup device agnostic code
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # For Mac M1/M2/M3 chips
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cuda


In [3]:
import torch
import torchvision
from torchvision.transforms import v2

In [26]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [17]:
!tar -xzvf "/kaggle/working/data/cifar-10-python.tar.gz" -C "/kaggle/working/"


cifar-10-batches-py/
cifar-10-batches-py/data_batch_4
cifar-10-batches-py/readme.html
cifar-10-batches-py/test_batch
cifar-10-batches-py/data_batch_3
cifar-10-batches-py/batches.meta
cifar-10-batches-py/data_batch_2
cifar-10-batches-py/data_batch_5
cifar-10-batches-py/data_batch_1


In [59]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(8, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [60]:


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum = 0.9)

In [61]:
import torch

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move model to GPU
net = net.to(device)

for epoch in range(15):

    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):

        # Get inputs and labels
        inputs, labels = data

        # Move them to GPU
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = net(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item()

        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

Using device: cuda
[1,  2000] loss: 2.174
[1,  4000] loss: 1.883
[1,  6000] loss: 1.685
[1,  8000] loss: 1.584
[1, 10000] loss: 1.503
[1, 12000] loss: 1.424
[2,  2000] loss: 1.358
[2,  4000] loss: 1.332
[2,  6000] loss: 1.292
[2,  8000] loss: 1.280
[2, 10000] loss: 1.250
[2, 12000] loss: 1.238
[3,  2000] loss: 1.159
[3,  4000] loss: 1.173
[3,  6000] loss: 1.170
[3,  8000] loss: 1.141
[3, 10000] loss: 1.147
[3, 12000] loss: 1.130
[4,  2000] loss: 1.065
[4,  4000] loss: 1.070
[4,  6000] loss: 1.072
[4,  8000] loss: 1.060
[4, 10000] loss: 1.047
[4, 12000] loss: 1.059
[5,  2000] loss: 0.962
[5,  4000] loss: 1.003
[5,  6000] loss: 0.995
[5,  8000] loss: 1.009
[5, 10000] loss: 0.999
[5, 12000] loss: 0.998
[6,  2000] loss: 0.909
[6,  4000] loss: 0.933
[6,  6000] loss: 0.953
[6,  8000] loss: 0.960
[6, 10000] loss: 0.948
[6, 12000] loss: 0.953
[7,  2000] loss: 0.872
[7,  4000] loss: 0.890
[7,  6000] loss: 0.888
[7,  8000] loss: 0.910
[7, 10000] loss: 0.894
[7, 12000] loss: 0.918
[8,  2000] loss

In [63]:
PATH = './cifar_net.pt'
torch.save(net.state_dict(), PATH)

In [64]:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [65]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 63 %
